In [1]:
# 05b_hallucination_honest.ipynb
# =============================================================================
# Notebook 05b — Honest Hallucination Comparison (reframed contribution)
#
# Reframes the contribution from "100% compliance" to "reduction of
# physiologically implausible recourse (hallucinations)".
#
# Fixes vs original nb06 Section 5:
#   - Scores violations against EXTERNAL fixed thresholds (WHO/KDRI),
#     independent of the range-generation formulas (no tautology).
#   - Reports guardrail effect at BOTH stages:
#       Guardrail_soft : CF generated under permitted_range, NO clip
#       Guardrail_hard : after deterministic hard-rule clip
#     so the reader sees what the clip actually contributes vs the search.
#   - Counts violations per-CF over ALL candidates (not just worst-per-case),
#     giving a rate rather than a hand-picked worst example.
#
# Violation types (external, absolute):
#   V_energy   : Energy_kcal < 800 kcal (clinical VLCD floor)  [G2-type]
#   V_conflict : sodium decreased AND (carb or sugar increased) [G3-type]
#   V_bmiwt    : BMI and Weight move in opposite directions     [G1-type]
#
# Input  : agent_config.pkl, df_final.pkl, model_{group}.pkl,
#          guardrail_ranges_v2.json
# Output : ../results/tables/hallucination_honest_percf.csv
#          ../results/tables/hallucination_honest_summary.csv
# =============================================================================

# %%
import json, joblib, warnings
import numpy as np, pandas as pd
import dice_ml

warnings.filterwarnings('ignore')

EXT_ENERGY_FLOOR = 800.0   # kcal, clinical VLCD lower boundary (external)
CHANGE_TOL = 1.0           # % change treated as "no change"

agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')
X_FEATURES      = agent_config['X_features']
VARY_FEATURES   = agent_config['vary_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {g: joblib.load(f'../results/tables/model_{g}.pkl') for g in AGEGROUP_CONFIG}
with open('../results/tables/guardrail_ranges_v2.json', encoding='utf-8') as f:
    GR = json.load(f)

# %%
# ## Violation detectors (external, absolute)

def v_energy(cf, orig):
    """1 = hallucination: energy below external clinical floor."""
    v = float(cf.get('Energy_kcal', orig.get('Energy_kcal', 0)))
    return int(v < EXT_ENERGY_FLOOR)

def v_conflict(cf, orig, tol=CHANGE_TOL):
    """1 = sodium cut but carb or sugar increased (implausible co-move)."""
    c_na = float(orig.get('Sodium_mg', 0)); v_na = float(cf.get('Sodium_mg', c_na))
    if not (c_na > 0 and (c_na - v_na)/c_na*100 > tol):
        return 0
    c_c = float(orig.get('Carb_g', 0));  v_c = float(cf.get('Carb_g', c_c))
    c_s = float(orig.get('Sugar_g', 0)); v_s = float(cf.get('Sugar_g', c_s))
    carb_up  = (v_c - c_c)/c_c*100 > tol if c_c > 0 else False
    sugar_up = (v_s - c_s)/c_s*100 > tol if c_s > 0 else False
    return int(carb_up or sugar_up)

def v_bmiwt(cf, orig, tol=CHANGE_TOL):
    """1 = BMI and Weight move in opposite directions (physically impossible)."""
    def chg(k):
        o = float(orig.get(k, 0)); v = float(cf.get(k, o))
        return (v - o)/o*100 if o != 0 else 0
    b, w = chg('BMI'), chg('Weight')
    if abs(b) <= tol or abs(w) <= tol:
        return 0
    return int(np.sign(b) != np.sign(w))

def clip_to(cf, ranges):
    out = dict(cf)
    for f, pair in ranges.items():
        if f in out and pair is not None:
            out[f] = float(np.clip(float(out[f]), float(pair[0]), float(pair[1])))
    return out

def build_permitted(ranges, df_ref, features):
    sr = {}
    for f in features:
        pair = ranges.get(f)
        if pair is None:
            continue
        lo, hi = float(pair[0]), float(pair[1])
        dlo = float(df_ref[f].quantile(0.05)); dhi = float(df_ref[f].quantile(0.95))
        sr[f] = [round(max(min(lo, dlo), 0.0), 4), round(max(hi, dhi), 4)]
    return sr

def gen_cfs(exp, query, permitted=None, n=4):
    kw = dict(total_CFs=n, desired_class=0, features_to_vary=VARY_FEATURES,
              proximity_weight=0.2, sparsity_weight=0.1)
    if permitted:
        kw['permitted_range'] = permitted
    try:
        cf = exp.generate_counterfactuals(query, **kw)
        return cf.cf_examples_list[0].final_cfs_df.to_dict('records') if cf else []
    except Exception:
        return []

# %%
records = []
for case_key, g in GR.items():
    grp = g['group']; mdl = models.get(grp)
    if mdl is None:
        continue
    orig = g['patient_profile']
    cfg = AGEGROUP_CONFIG[grp]
    age = 0.0 if cfg['age_min'] < 60 else 1.0
    dref = df_final[(df_final['AgeGroup']==age) & (df_final['Sex']==cfg['sex_code'])].copy()
    query = pd.DataFrame([orig])[X_FEATURES]

    d = dice_ml.Data(dataframe=df_final.copy().astype(float)[X_FEATURES+[TARGET_COL]],
                     continuous_features=X_FEATURES, outcome_name=TARGET_COL)
    m = dice_ml.Model(model=mdl, backend='sklearn')
    exp = dice_ml.Dice(d, m, method='genetic')

    hard_ranges = g['hardrule_ranges']
    permitted = build_permitted(g['final_ranges'], dref, X_FEATURES)

    # Pure DiCE (no constraints)
    for cf in gen_cfs(exp, query, None):
        records.append({'CaseKey':case_key,'Group':grp,'Condition':'PureDiCE',
                        'V_energy':v_energy(cf,orig),'V_conflict':v_conflict(cf,orig),
                        'V_bmiwt':v_bmiwt(cf,orig)})

    # Guardrail soft (permitted_range, no clip)
    soft_cfs = gen_cfs(exp, query, permitted)
    for cf in soft_cfs:
        records.append({'CaseKey':case_key,'Group':grp,'Condition':'Guardrail_soft',
                        'V_energy':v_energy(cf,orig),'V_conflict':v_conflict(cf,orig),
                        'V_bmiwt':v_bmiwt(cf,orig)})

    # Guardrail hard (same CFs, then deterministic clip)
    for cf in soft_cfs:
        cfh = clip_to(cf, hard_ranges)
        records.append({'CaseKey':case_key,'Group':grp,'Condition':'Guardrail_hard',
                        'V_energy':v_energy(cfh,orig),'V_conflict':v_conflict(cfh,orig),
                        'V_bmiwt':v_bmiwt(cfh,orig)})
    print(f"  [{case_key}] done")

df = pd.DataFrame(records)
df.to_csv('../results/tables/hallucination_honest_percf.csv', index=False, encoding='utf-8-sig')

# %%
# Summary: violation RATE (% of CF candidates with each violation type)
summary = (df.groupby('Condition')[['V_energy','V_conflict','V_bmiwt']].mean()*100).round(1)
order = ['PureDiCE','Guardrail_soft','Guardrail_hard']
summary = summary.reindex([c for c in order if c in summary.index])
summary['n_CF'] = df.groupby('Condition').size().reindex(summary.index)
summary.to_csv('../results/tables/hallucination_honest_summary.csv', encoding='utf-8-sig')
print("\n=== Hallucination violation rate (%, per-CF, external scoring) ===")
print(summary.to_string())

# McNemar-style case-level: fraction of cases with >=1 violation
print("\n=== Case-level: fraction of cases with >=1 violation ===")
for cond in order:
    sub = df[df['Condition']==cond]
    by_case = sub.groupby('CaseKey')[['V_energy','V_conflict','V_bmiwt']].max()
    print(f"  {cond:16s}: "
          f"energy {int(by_case['V_energy'].sum())}/12  "
          f"conflict {int(by_case['V_conflict'].sum())}/12  "
          f"bmiwt {int(by_case['V_bmiwt'].sum())}/12")

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.40it/s]


  [MiddleAged_Male_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.64it/s]


  [MiddleAged_Male_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.39it/s]


...

  [Older_Female_case1] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.79it/s]


  [Older_Female_case2] done


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.68it/s]

  [Older_Female_case3] done

=== Hallucination violation rate (%, per-CF, external scoring) ===
                V_energy  V_conflict  V_bmiwt  n_CF
Condition                                          
PureDiCE             4.2        31.2      8.3    48
Guardrail_soft       2.1        42.6      8.5    47
Guardrail_hard       0.0         0.0      0.0    47

=== Case-level: fraction of cases with >=1 violation ===
  PureDiCE        : energy 2/12  conflict 8/12  bmiwt 2/12
  Guardrail_soft  : energy 1/12  conflict 10/12  bmiwt 2/12
  Guardrail_hard  : energy 0/12  conflict 0/12  bmiwt 0/12
